In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
os.listdir(path) #to find the files INSIDE the folder, so you can prepare it for Pandas to read it
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)
print(f"Dataset shape: {df_delivery.shape}") # Data has 1663 samples and 9 features.

In [ ]:
# Task 2: Write your code here:
df_delivery.head(20) # We have 4 categorical variables to encode, I will use label encoding so I won't add too many dimensions
# Numerical features needs scaling too, nothing too crazy but to be safe, DON'T SCALE THE TARGET
# Courier_Experience_yrs has some NaN values, could be a data collection error or no experience? idk, I think I'll check the count first, if they were a little I'll just drop rows ig

In [ ]:
# Task 3: Write your code here:
df_delivery.info() # We have 4 categorical variables as I mentioned above, ENCODING NEEDED
# We have even more NaN values in Weather, Traffic_Level, Time_of_Day, and Delivery_Time on top of the Courier_Experience_yrs I detected above
# Let me count them to see what to do

In [ ]:
# Task 4: Write your code here:
df_delivery.describe() #

In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)

df_delivery['Delivery_Time'].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({'Delivery_Time'})")
plt.xlabel('Delivery_Time')
plt.ylabel("Time in Minutes")
plt.grid(False)

plt.show()

# Data looks fairly normal but a lil right skewed, possible outliers. ATTENTION HERE

In [ ]:
# Task 1: Write your code here:
df_delivery = df_delivery.drop('Order_ID', axis=1)
df_delivery.head() # No ID, good.

In [ ]:
# Task 2: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery)
# We have over 1600 samples, I think I'll drop the rows of Weather, Traffic_Level, Time_of_Day, Courier_Experience_yrs. Also Delivery_Time because it's the target
# FYI, I run the cell again after deleting duplicates, that's why they are less now.

df_clean = df_delivery.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])
df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())
print(f"Dataset shape: {df_clean.shape}") # Data originally had 1663 samples and 9 features.
df_clean.head()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery)
# I removes the duplicates that's why it's 0 now :)

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print('data before encoding:\n', categorical_cols) #show before encoding

for col in categorical_cols:
    le = OneHotEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()


In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
# Define features (X) and target (y)
features = df_clean.columns.drop('Delivery_Time')
print(features)


X = df_clean[features]
print(X.head())
print('========================')
y = df_clean['Delivery_Time']
print(y.head())

# I wanna split the data before scaling, to prevent data leakage, best practices and stuff, so I'll jump some steps if that's ok whoever you are :)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # I will do fit_transform here, and I'll only transform and X_test so the statistcs used to scale the data won't let the model cheat
X_test_scaled = scaler.transform(X_test) # Only transform here

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head()


In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)


# No need because the target is a contnious value, and since it's a regression problem and NOT classification, no imbalnce check needed

In [ ]:
# Task 1: Write your code here:
# Already done in previous cells to prevent data leakage, I splitted before scaling

In [ ]:
# Task 2,3,4,5: Write your code here:

model = RandomForestRegressor(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")


kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)


print(f"5-Fold CV Results:")
print(f"MAE: {mae_scores.mean():,.2f} minutes")


# Now we compare what we just got with the baseline to check if our model beat it
baseline_pred = np.full_like(y_test, y_train.median(), dtype=float) #we are saying, create a thing called baseline_pred that has the shape of y_test but filled with the mean of y_train
#And make sure you only use y_train.mean(), never use y_val


mae_baseline = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline mae (median predictor): {mae_baseline:.4f}")

# My model is better


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: